# Notebook 04 - Elasticity gradient (R4) - reproduces B&F Fig. 7

Faithful port of `elasticity_gradient.m` / `eg.m`.

For each of sectors [10, 20, 23], apply an idiosyncratic TFP shock (A[sec]=1.1) and sweep ONE elasticity
over (0.015, 0.99) while holding the other two at their baseline (0.015). Records:

  * the **real (CES-welfare) GDP** from `eg.m`  (C = sum_i L_i p_i A_i^((e-1)/e) a_i^(1/e) y_i^(1/e) (1/L_i)^(1/e)),
    using the *swept* elasticity in the exponent for the epsilon sweep, baseline otherwise;
  * the Domar-weighted **mean price** from `eg.m`  (sum_i beta_i * p_i^(-sigma), sigma swept only for the sigma sweep).

Continuation (warm-start) is used across the grid so the Newton solver tracks the root near the degenerate extremes.
The six result matrices are exported as CSV; figures are generated downstream in Python from those CSVs.

In [ ]:
const NB_DIR = @__DIR__
const PKG = joinpath(NB_DIR, "..")
const DATA_DIR = joinpath(PKG, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(PKG, "results")
mkpath(RESULTS_DIR)
push!(LOAD_PATH, joinpath(PKG, "src"))
include(joinpath(PKG, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader

In [ ]:
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
res = run_elasticity_gradient(data; sectors=[10, 20, 23], shock_val=1.1,
                              grid=range(0.015, 0.99; length=100))
println("done. gdp_eps size = ", size(res.gdp_eps), "; grid length = ", length(res.grid))

In [ ]:
function write_grid_matrix(path, M, sectors, grid)
    open(path, "w") do io
        print(io, "sector,")
        println(io, join(round.(grid, digits=4), ","))
        for i in 1:size(M, 1)
            print(io, sectors[i], ",")
            println(io, join(M[i, :], ","))
        end
    end
end
for (name, M) in [("elasticity_gdp_epsilon", res.gdp_eps),
                  ("elasticity_gdp_theta",   res.gdp_theta),
                  ("elasticity_gdp_sigma",   res.gdp_sigma),
                  ("elasticity_mp_epsilon",  res.mp_eps),
                  ("elasticity_mp_theta",    res.mp_theta),
                  ("elasticity_mp_sigma",    res.mp_sigma)]
    write_grid_matrix(joinpath(RESULTS_DIR, name * ".csv"), M, res.sectors, res.grid)
end
println("exported 6 elasticity CSVs to ", RESULTS_DIR)